# parallel-downloader: notebook demo

Three small cells demonstrating the library's three integration surfaces. Open in [kotlin-jupyter](https://github.com/Kotlin/kotlin-jupyter) (`pip install kotlin-jupyter-kernel` or via Conda) with the project's runtime jar on the classpath:

```bash
./gradlew installDist
jupyter notebook docs/notebook-demo.ipynb
```

Each cell is independent. Replace the URL placeholder with one you have HTTP access to.

## 1. The `Flow<ProgressEvent>` API

Pull-style consumption: collect events from a coroutine `Flow`. Use this when your caller is already coroutine-shaped and prefers `flowOn` / `collect` over a push callback. Cancelling the collector cancels the download via structured concurrency.

In [ ]:
@file:DependsOn("build/libs/parallel-downloader-1.0.0.jar")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core:1.9.0")

import com.example.downloader.*
import com.example.downloader.http.JdkHttpRangeFetcher
import kotlinx.coroutines.flow.collect
import kotlinx.coroutines.runBlocking
import java.net.URL
import java.nio.file.Path

val downloader = FileDownloader(JdkHttpRangeFetcher())
val url = URL("http://localhost:8090/big.bin")  // serve via :jettyFixture or any local server
val dest = Path.of("/tmp/big.bin")

runBlocking {
    downloader.downloadAsFlow(url, dest).collect { event ->
        when (event) {
            is ProgressEvent.Started -> println("started: total=${event.total} bytes")
            is ProgressEvent.Progress -> println("progress: ${event.downloaded}/${event.total}")
            is ProgressEvent.ChunkComplete -> println("chunk ${event.index} done")
            is ProgressEvent.Finished -> println("finished: ${event.result}")
        }
    }
}

## 2. The `downloadConfig { ... }` builder DSL

Idiomatic Kotlin trailing-lambda construction. Set chunk size, parallelism, rate limit, and resume mode in one block. Defaults are sensible (8 MiB chunks, parallelism 8, no resume); the DSL is for callers who want to deviate.

In [ ]:
import com.example.downloader.*

// `8.MiB` and `5L * 1024L * 1024L` are equivalent; the size extensions are for readability.
val cfg = downloadConfig {
    chunkSize = 8.MiB
    parallelism = 16
    rateLimitBytesPerSec = 50L * 1024L * 1024L  // 50 MiB/s cap
    resume = true                                // reuse an existing .part file if validators match
    overwriteExisting = false                    // fail if dest already exists
}

println("chunk=${cfg.chunkSize} parallelism=${cfg.parallelism} resume=${cfg.resume}")

## 3. A custom `Telemetry` implementation

The `Telemetry` interface is the supported metric-collection seam. Its method signatures take **counters, byte counts, chunk indices, and retry numbers** - never URL hosts, paths, or error strings. The privacy-typed surface means an implementation can do whatever it likes with what it receives, but only ever receives non-identifying data.

Below: a toy in-memory telemetry that aggregates per-download stats. For a JUL-backed reference, see [`com.example.downloader.telemetry.LoggingTelemetry`](../src/main/kotlin/com/example/downloader/telemetry/LoggingTelemetry.kt).

In [ ]:
import com.example.downloader.*
import com.example.downloader.http.JdkHttpRangeFetcher
import kotlinx.coroutines.runBlocking
import java.net.URL
import java.nio.file.Path
import java.util.concurrent.atomic.AtomicInteger
import java.util.concurrent.atomic.AtomicLong
import kotlin.time.Duration

class CountingTelemetry : Telemetry {
    private val chunks = AtomicInteger(0)
    private val bytes = AtomicLong(0L)
    private val retries = AtomicInteger(0)

    override fun onChunkComplete(chunkIndex: Int, chunkBytes: Long) {
        chunks.incrementAndGet()
        bytes.addAndGet(chunkBytes)
    }

    override fun onTransientFailure(retryAttempt: Int) {
        retries.incrementAndGet()
    }

    override fun onDownloadComplete(totalBytes: Long, elapsed: Duration, chunks: Int, retries: Int) {
        println("summary: chunks=$chunks bytes=$totalBytes elapsed=$elapsed retries=$retries")
    }
}

val telemetry = CountingTelemetry()
val cfg = downloadConfig {
    chunkSize = 8.MiB
    parallelism = 8
    this.telemetry = telemetry
}
val downloader = FileDownloader(JdkHttpRangeFetcher())
runBlocking {
    val result = downloader.download(
        URL("http://localhost:8090/big.bin"),
        Path.of("/tmp/big.bin"),
        cfg,
    )
    println("result: $result")
}

## See also

- [`README.md`](../README.md): quick start and CLI usage
- [`docs/DESIGN.md`](DESIGN.md): architecture, concurrency model, design forks
- [`docs/RFC-COMPLIANCE.md`](RFC-COMPLIANCE.md): RFC 9110 coverage matrix
- [`docs/STORY-CONCURRENCY-FIX.md`](STORY-CONCURRENCY-FIX.md): the `limitedParallelism` debugging story